# TUGAS 2

## 1. Install library terlebih dahulu

In [53]:
pip install pandas openpyxl numpy scikit-learn

## 2. Import library

In [54]:
import pandas as pd
import numpy as np
import re

from scipy.sparse import csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 3. Membaca dataset

In [55]:
df = pd.read_excel("dataset_detik_200_berita.xlsx")

df.head()

,id,isi_berita,label
0,1,Men's World Tennis Championship 2026 sudah mem...,sport
1,2,Barra Ghaisan Zeinatma menunjukkan perkembanga...,sport
2,3,Marco Bezzecchi tak sabar menghadapi MotoGP Sa...,sport
3,4,Marc Marquez masih harus beradaptasi dengan ko...,sport
4,5,Pebalap Mercedes GP Kimi Antonelli tampil luar...,sport


### Cek jumlah data:

In [56]:
print("Jumlah data :", len(df))
print("\nNama kolom:")
print(df.columns)

print("\nJumlah data per label:")
print(df["label"].value_counts())

Jumlah data : 200

Nama kolom:
Index(['id', 'isi_berita', 'label'], dtype='object')

Jumlah data per label:
label
sport      100
finance    100
Name: count, dtype: int64


## 4. Mengubah label menjadi numerik

In [57]:
df["label_num"] = df["label"].map({
    "sport": 1,
    "finance": 0
})

df[["id", "label", "label_num"]].head()

,id,label,label_num
0,1,sport,1
1,2,sport,1
2,3,sport,1
3,4,sport,1
4,5,sport,1


In [58]:
print(df["label_num"].value_counts())

label_num
1    100
0    100
Name: count, dtype: int64


## 5. Menghitung jumlah kata semua berita

In [59]:
df["jumlah_kata_asli"] = df["isi_berita"].astype(str).apply(
    lambda x: len(x.split())
)

df[["id", "jumlah_kata_asli"]].head()

total_kata = df["jumlah_kata_asli"].sum()

print("Total seluruh kata :", total_kata)

print(df["jumlah_kata_asli"].describe())

Total seluruh kata : 61542
count     200.000000
mean      307.710000
std       147.114955
min         1.000000
25%       235.750000
50%       289.000000
75%       347.750000
max      1037.000000
Name: jumlah_kata_asli, dtype: float64


### melihat jumlah kata setiap berita

In [60]:
df[[
    "id",
    "label",
    "jumlah_kata_asli"
]]

,id,label,jumlah_kata_asli
0,1,sport,297
1,2,sport,237
2,3,sport,256
3,4,sport,243
4,5,sport,224
...,...,...,...
195,196,finance,45
196,197,finance,429
197,198,finance,381
198,199,finance,309


## 6. Kamus kata tidak baku

In [61]:
kamus_tidak_baku = {
    "gak": "tidak",
    "nggak": "tidak",
    "ga": "tidak",
    "enggak": "tidak",
    "yg": "yang",
    "dgn": "dengan",
    "utk": "untuk",
    "krn": "karena",
    "kalo": "kalau",
    "kalok": "kalau",
    "aja": "saja",
    "udah": "sudah",
    "sdh": "sudah",
    "blm": "belum",
    "tdk": "tidak",
    "dr": "dari",
    "dlm": "dalam",
    "jd": "jadi",
    "bgt": "banget",
    "tp": "tetapi",
    "tapi": "tetapi",
    "karna": "karena",
    "trus": "terus",
    "kmrn": "kemarin",
    "dpt": "dapat",
    "hrs": "harus",
    "sm": "sama",
    "sy": "saya"
}

## 7. Kamus bahasa asing

In [62]:
kamus_asing = {

    # SPORT
    "rider": "pembalap",
    "race": "balapan",
    "racing": "balap",
    "team": "tim",
    "coach": "pelatih",
    "player": "pemain",
    "match": "pertandingan",
    "winner": "pemenang",
    "season": "musim",
    "training": "latihan",
    "game": "pertandingan",
    "games": "pertandingan",
    "manager": "manajer",
    "champion": "juara",
    "championship": "kejuaraan",
    "league": "liga",
    "score": "skor",
    "goal": "gol",
    "final": "final",

    # FINANCE
    "finance": "keuangan",
    "financial": "keuangan",
    "market": "pasar",
    "stock": "saham",
    "stocks": "saham",
    "sale": "penjualan",
    "price": "harga",
    "business": "bisnis",
    "company": "perusahaan",
    "investment": "investasi",
    "investor": "investor",
    "banking": "perbankan",
    "bank": "bank",
    "economy": "ekonomi",
    "economic": "ekonomi",
    "growth": "pertumbuhan",
    "profit": "keuntungan",
    "loss": "kerugian",
    "revenue": "pendapatan"
}

## 8. Fungsi preprocessing


Berikut adalah penjelasan detail dari setiap tahapan yang ada di dalam kode:

* Mengubah Tipe Data (str(text)): Memastikan bahwa masukan yang diproses bertipe data string (teks), guna menghindari eror jika ada data kosong (NaN/null) atau bertipe angka.
* Case Folding (text.lower()): Mengubah semua huruf menjadi huruf kecil (lowercase) agar kata yang sama tidak dianggap berbeda hanya karena perbedaan huruf kapital (misal: "Buku" dan "buku" akan dianggap sama).
* Hapus URL & Email: Menghilangkan tautan situs web (seperti https://... atau www.) dan alamat email menggunakan Regular Expression (regex) karena informasi ini biasanya tidak diperlukan dalam analisis sentimen atau teks umum.
* Hapus Angka: Menghilangkan semua karakter angka (0-9).
* Hapus Tanda Baca, Simbol, dan Emoticon: Menghapus semua karakter selain huruf alfabet (a-z, A-Z) dan karakter beraksen (À-ÿ). Karakter seperti !, @, #, $, serta emoji akan diubah menjadi spasi.
* Hapus Spasi Berlebih (.strip()): Mengompres spasi ganda atau berlebih akibat proses penghapusan sebelumnya menjadi satu spasi saja, lalu memotong spasi di awal dan akhir teks.
* Tokenisasi (text.split()): Memecah kalimat menjadi potongan kata-kata individu (token) berdasarkan spasi.
* Normalisasi Teks (Kamus Baku & Asing):
* Mengubah kata tidak baku: Memeriksa setiap kata ke dalam kamus_tidak_baku (misal: mengubah "bgt" menjadi "banget", "yg" menjadi "yang").
   * Menerjemahkan bahasa asing: Memeriksa kata ke dalam kamus_asing untuk diterjemahkan ke bahasa Indonesia (misal: mengubah "online" menjadi "daring").
* Penggabungan Kembali (" ".join(hasil)): Menyatukan kembali token kata-kata yang sudah bersih dan baku tersebut menjadi satu string kalimat utuh.






In [63]:
def preprocessing(text):

    # Pastikan berupa string
    text = str(text)

    # =========================
    # CASE FOLDING
    # =========================
    text = text.lower()


    # =========================
    # HAPUS URL
    # =========================
    text = re.sub(
        r'https?://\S+|www\.\S+',
        ' ',
        text
    )


    # =========================
    # HAPUS EMAIL
    # =========================
    text = re.sub(
        r'\S+@\S+',
        ' ',
        text
    )


    # =========================
    # HAPUS ANGKA
    # =========================
    text = re.sub(
        r'\d+',
        ' ',
        text
    )


    # =========================
    # HAPUS TANDA BACA,
    # SIMBOL DAN EMOTICON
    # =========================

    text = re.sub(
        r'[^a-zA-ZÀ-ÿ\s]',
        ' ',
        text
    )


    # =========================
    # HAPUS SPASI BERLEBIH
    # =========================

    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()


    # =========================
    # TOKENISASI
    # =========================

    kata = text.split()


    hasil = []

    for token in kata:

        # Membakukan kata
        if token in kamus_tidak_baku:
            token = kamus_tidak_baku[token]


        # Bahasa asing -> Indonesia
        if token in kamus_asing:
            token = kamus_asing[token]


        hasil.append(token)


    return " ".join(hasil)

## 9. Terapkan preprocessing ke semua berita

kode tersebut berfungsi untuk menerapkan fungsi preprocessing ke seluruh baris data pada DataFrame Pandas, lalu menampilkan sampel hasilnya untuk memastikan teks sudah bersih.

In [64]:
df["berita_clean"] = df["isi_berita"].apply(preprocessing)

df[[
    "id",
    "isi_berita",
    "berita_clean"
]].head()

,id,isi_berita,berita_clean
0,1,Men's World Tennis Championship 2026 sudah mem...,men s world tennis kejuaraan sudah memasuki se...
1,2,Barra Ghaisan Zeinatma menunjukkan perkembanga...,barra ghaisan zeinatma menunjukkan perkembanga...
2,3,Marco Bezzecchi tak sabar menghadapi MotoGP Sa...,marco bezzecchi tak sabar menghadapi motogp sa...
3,4,Marc Marquez masih harus beradaptasi dengan ko...,marc marquez masih harus beradaptasi dengan ko...
4,5,Pebalap Mercedes GP Kimi Antonelli tampil luar...,pebalap mercedes gp kimi antonelli tampil luar...


## 10. Hitung jumlah kata setelah preprocessing

kode tersebut berfungsi untuk menghitung jumlah kata setelah teks dibersihkan, lalu menampilkan perbandingannya dengan jumlah kata pada teks asli.

In [65]:
df["jumlah_kata_clean"] = df["berita_clean"].apply(
    lambda x: len(x.split())
)

df[[
    "id",
    "jumlah_kata_asli",
    "jumlah_kata_clean"
]].head()

,id,jumlah_kata_asli,jumlah_kata_clean
0,1,297,286
1,2,237,227
2,3,256,252
3,4,243,242
4,5,224,215


### Total:

In [66]:
print(
    "Total kata sebelum preprocessing :",
    df["jumlah_kata_asli"].sum()
)

print(
    "Total kata setelah preprocessing :",
    df["jumlah_kata_clean"].sum()
)

Total kata sebelum preprocessing : 61542
Total kata setelah preprocessing : 60166


## 11. Ekstrak seluruh kata unik

 kode tersebut berfungsi untuk menghitung total seluruh kata dan total kosakata unik (vocab) dari seluruh dokumen berita yang telah dibersihkan.

In [67]:
semua_kata = []

for berita in df["berita_clean"]:
    semua_kata.extend(
        berita.split()
    )

kata_unik = sorted(
    set(semua_kata)
)

print(
    "Jumlah seluruh kata:",
    len(semua_kata)
)

print(
    "Jumlah kata unik:",
    len(kata_unik)
)

Jumlah seluruh kata: 60166
Jumlah kata unik: 6919


In [68]:
kata_unik[:100]

['a',
 'aadi',
 'aan',
 'abadi',
 'abal',
 'abang',
 'abdi',
 'abdul',
 'aberdeen',
 'absen',
 'absennya',
 'abu',
 'abullah',
 'acara',
 'access',
 'account',
 'accurate',
 'acd',
 'aceh',
 'acid',
 'acosta',
 'activ',
 'activation',
 'activities',
 'acuan',
 'ada',
 'adain',
 'adalah',
 'adanya',
 'adaptasi',
 'adapun',
 'adhitya',
 'adianto',
 'adik',
 'adil',
 'adininggar',
 'administrasi',
 'administratif',
 'adna',
 'adopsi',
 'adp',
 'adriatik',
 'adu',
 'aduan',
 'advisory',
 'advokat',
 'aerodinamika',
 'aeronautika',
 'aesi',
 'af',
 'aff',
 'afrianto',
 'afrika',
 'ag',
 'agak',
 'agama',
 'agar',
 'agency',
 'agenda',
 'agent',
 'agraria',
 'agreement',
 'agregat',
 'agresif',
 'agresivitas',
 'agrinas',
 'agu',
 'agung',
 'agus',
 'agusman',
 'agustian',
 'agustus',
 'ahi',
 'ahmad',
 'ahmed',
 'ahren',
 'ahsanurrohim',
 'ai',
 'aichi',
 'air',
 'airlangga',
 'airmen',
 'airnav',
 'airport',
 'airports',
 'ajaib',
 'ajakan',
 'ajang',
 'ajaran',
 'akademik',
 'akademisi',


In [69]:
df_kata_unik = pd.DataFrame({
    "kata_unik": kata_unik
})

df_kata_unik.head(20)

,kata_unik
0,a
1,aadi
2,aan
3,abadi
4,abal
5,abang
6,abdi
7,abdul
8,aberdeen
9,absen


In [70]:
df_kata_unik.to_excel(
    "kata_unik.xlsx",
    index=False
)

## 12. Membagi data 160 training dan 40 testing

 kode tersebut berfungsi untuk membagi dataset menjadi dua bagian, yaitu Data Latih (Training Set) dan Data Uji (Testing Set) secara adil dan proporsional sebelum dimasukkan ke dalam model Machine Learning.

In [71]:
X_train_text, X_test_text, y_train, y_test = train_test_split(

    df["berita_clean"],
    df["label_num"],

    test_size=40,

    random_state=42,

    stratify=df["label_num"]
)

In [72]:
print(
    "Jumlah training:",
    len(X_train_text)
)

print(
    "Jumlah testing:",
    len(X_test_text)
)

Jumlah training: 160
Jumlah testing: 40


In [73]:
print("TRAINING")
print(y_train.value_counts())

print("\nTESTING")
print(y_test.value_counts())

TRAINING
label_num
0    80
1    80
Name: count, dtype: int64

TESTING
label_num
0    20
1    20
Name: count, dtype: int64


## 13. Representasi TF-IDF

kode tersebut berfungsi untuk mengubah teks berita menjadi matriks angka menggunakan metode TF-IDF (Term Frequency-Inverse Document Frequency), sekaligus melakukan seleksi kata (fitur) berdasarkan frekuensi kemunculannya.

In [74]:
tfidf = TfidfVectorizer(
    min_df=2,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(
    X_train_text
)

X_test_tfidf = tfidf.transform(
    X_test_text
)

In [75]:
nama_fitur = tfidf.get_feature_names_out()

print(
    "Jumlah fitur TF-IDF:",
    len(nama_fitur)
)

Jumlah fitur TF-IDF: 3069


## 14. Melihat matriks TF-IDF

In [76]:
tfidf_train_df = pd.DataFrame(
    X_train_tfidf.toarray(),
    columns=nama_fitur
)

tfidf_train_df.head()

,abdul,absen,abu,acara,acd,aceh,acosta,ada,adalah,adanya,...,youtube,yudhi,yusrian,yusuf,zapp,zarco,zona,zoom,zulhas,zulkifli
0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,...,0.0,0.086771,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
1,0.049537,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,...,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
2,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.031766,0.017973,0.026316,...,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
3,0.163842,0.0,0.0,0.0,0.0,0.0,0.0,0.023336,0.026406,0.000000,...,0.0,0.000000,0.0,0.0,0.101903,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,...,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0


In [77]:
tfidf_train_df["label"] = y_train.reset_index(
    drop=True
)

tfidf_train_df.head()

,abdul,absen,abu,acara,acd,aceh,acosta,ada,adalah,adanya,...,yudhi,yusrian,yusuf,zapp,zarco,zona,zoom,zulhas,zulkifli,label
0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,...,0.086771,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0
1,0.049537,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,1
2,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.031766,0.017973,0.026316,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0
3,0.163842,0.0,0.0,0.0,0.0,0.0,0.0,0.023336,0.026406,0.000000,...,0.000000,0.0,0.0,0.101903,0.0,0.0,0.0,0.0,0.0,1
4,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,1


## 15. Simpan data TF-IDF

kode tersebut dibagi menjadi dua bagian utama: penyimpanan hasil pembobotan TF-IDF ke dalam file Excel dan perhitungan bobot fitur baru menggunakan metode ICSDF (Inverse Class Space Document Frequency).
Berikut adalah penjelasan detail untuk setiap alur kerjanya:
## 1. Menyimpan Matriks TF-IDF ke File Excel

* Menyimpan Data Latih (tfidf_train_df.to_excel(...)): Baris pertama mengekspor DataFrame data latih (yang diasumsikan sudah dibuat sebelumnya) ke file bernama data_tfidf_training.xlsx tanpa menyertakan indeks baris (index=False).
* Membuat DataFrame Uji (pd.DataFrame(...)): Mengubah matriks renggang (sparse matrix) X_test_tfidf menjadi matriks padat (array biasa) menggunakan .toarray(), lalu menjadikannya kolom-kolom dengan nama kata yang sesuai (columns=nama_fitur).
* Menggabungkan Label Uji (tfidf_test_df["label"] = ...): Menambahkan kolom "label" berisi y_test. Fungsi .reset_index(drop=True) digunakan agar urutan indeks label selaras dengan DataFrame baru dan tidak berantakan.
* Menyimpan Data Uji (tfidf_test_df.to_excel(...)): Mengekspor hasil akhir matriks data uji beserta labelnya ke file data_tfidf_testing.xlsx.

------------------------------
## 2. Perhitungan Metode ICSDF (Inverse Class Space Document Frequency)
ICSDF adalah metode seleksi atau pembobotan fitur berbasis kelas. Tujuannya adalah mencari kata-kata yang unik dan hanya muncul di kelas/kategori tertentu saja, serta mengabaikan kata yang muncul merata di semua kelas.
Alur algoritmanya berjalan sebagai berikut:

* Inisialisasi Variabel:
* jumlah_kelas: Menghitung berapa banyak kategori unik yang ada di data latih (misal: 2 kelas untuk Hoaks dan Fakta).
   * class_space_density: Membuat array kosong berisi angka 0 sepanjang jumlah fitur (kata) untuk menampung total densitas kata di setiap kelas.
* Perulangan per Kelas (for kelas in ...):
* mask = (y_train_array == kelas): Membuat penyaring untuk mengambil baris data yang masuk ke dalam kelas tertentu saja.
   * X_class = X_train_tfidf[mask]: Memisahkan dokumen yang hanya termasuk dalam kelas tersebut.
   * document_frequency_class: Menghitung di berapa banyak dokumen dalam kelas tersebut suatu kata muncul (di mana nilai TF-IDF > 0). .A1 digunakan untuk meratakan (flatten) hasilnya menjadi array 1 dimensi.
   * class_density: Menghitung kerapatan kata di kelas tersebut dengan rumus:
   $$\text{Kerapatan} = \frac{\text{Jumlah dokumen di kelas yang mengandung kata}}{\text{Total seluruh dokumen di kelas tersebut}}$$
   * class_space_density += class_density: Akumulasi nilai kerapatan kata dari seluruh kelas yang ada.
* Rumus Inti ICSDF (np.log(...)):
* epsilon = 1e-12: Angka yang sangat kecil untuk menghindari eror pembagian dengan nol ($0$) atau nilai $\log(0)$ yang tidak terdefinisi.
   * Formula yang diterapkan adalah:
   $$\text{ICSDF} = \log\left(\frac{\text{Jumlah Kelas}}{\text{Total Kerapatan Kata di Semua Kelas}}\right)$$
   * Logika Rumus: Jika suatu kata muncul sangat sering di semua kelas, nilai class_space_density-nya akan besar (mendekati jumlah kelas), sehingga hasil pembagiannya mendekati 1 dan nilai $\log(1) = 0$ (bobotnya rendah). Sebaliknya, jika suatu kata hanya muncul di satu kelas tertentu saja, nilai pembaginya kecil, membuat nilai $\log$-nya besar (bobotnya tinggi).
* Menampilkan Hasil (df_icsdf.sort_values(...)):
* Memasukkan hasil perhitungan bobot ke dalam DataFrame dengan kolom "kata" dan "ICSDF".
   * Mengurutkan kata dari nilai ICSDF tertinggi ke terendah (ascending=False) dan menampilkan 20 kata teratas menggunakan .head(20). Kata-kata inilah yang dianggap paling diskriminatif (paling kuat) dalam membedakan antar-kelas berita.





In [78]:
tfidf_train_df.to_excel(
    "data_tfidf_training.xlsx",
    index=False
)

In [79]:
tfidf_test_df = pd.DataFrame(
    X_test_tfidf.toarray(),
    columns=nama_fitur
)

tfidf_test_df["label"] = y_test.reset_index(
    drop=True
)

tfidf_test_df.to_excel(
    "data_tfidf_testing.xlsx",
    index=False
)

In [80]:
features = np.array(
    tfidf.get_feature_names_out()
)

jumlah_kelas = y_train.nunique()

class_space_density = np.zeros(
    len(features)
)

y_train_array = y_train.to_numpy()

In [81]:
for kelas in sorted(
    y_train.unique()
):

    mask = (
        y_train_array == kelas
    )

    X_class = X_train_tfidf[
        mask
    ]


    # Berapa dokumen kelas tersebut
    # mengandung sebuah kata

    document_frequency_class = (
        (X_class > 0)
        .sum(axis=0)
        .A1
    )


    jumlah_dokumen_class = (
        X_class.shape[0]
    )


    class_density = (
        document_frequency_class
        /
        jumlah_dokumen_class
    )


    class_space_density += (
        class_density
    )

In [82]:
epsilon = 1e-12

icsdf = np.log(

    (jumlah_kelas + epsilon)

    /

    (class_space_density + epsilon)
)

In [83]:
df_icsdf = pd.DataFrame({

    "kata": features,

    "ICSDF": icsdf
})

df_icsdf.sort_values(
    "ICSDF",
    ascending=False
).head(20)

,kata,ICSDF
11,adik,4.382027
3068,zulkifli,4.382027
3067,zulhas,4.382027
14,adu,4.382027
3066,zoom,4.382027
15,aeronautika,4.382027
3064,zarco,4.382027
5,aceh,4.382027
17,agak,4.382027
1842,mobilnya,4.382027


## 17. TF-IDF × ICSDF

kode tersebut berfungsi untuk menerapkan bobot ICSDF ke dalam matriks TF-IDF (proses pembobotan gabungan atau kombinasi hybrid TF-IDF + ICSDF) baik pada data latih maupun data uji, kemudian mengembalikannya ke dalam bentuk matriks renggang (sparse matrix).
Berikut adalah penjelasan detail untuk setiap alur kerjanya:

* Mengalikan Matriks dengan Bobot ICSDF (.multiply(icsdf))
* X_train_tfidf.multiply(icsdf): Setiap nilai TF-IDF dari suatu kata pada data latih akan dikalikan secara langsung dengan nilai bobot ICSDF yang sudah dihitung sebelumnya untuk kata tersebut.
   * X_test_tfidf.multiply(icsdf): Proses yang sama diterapkan pada data uji. Data uji dikalikan menggunakan nilai icsdf yang dipelajari dari data latih agar konsisten dan menghindari kebocoran data (data leakage).
   * Tujuan: Kata-kata yang memiliki skor ICSDF tinggi (kata yang sangat khas pada kelas tertentu) nilainya akan semakin diperkuat, sedangkan kata-kata yang skor ICSDF-nya rendah (kata umum yang muncul di semua kelas) nilainya akan diperkecil atau mendekati nol.
* Mengonversi Kembali ke Matriks Renggang (csr_matrix(...))
* csr_matrix(X_train_icsdf): Mengonversi kembali hasil perkalian menjadi format Compressed Sparse Row (CSR) dari pustaka .
   * Tujuan: Karena sebagian besar nilai dalam matriks teks berbobot ini adalah angka nol (0), format csr_matrix akan menghemat penggunaan memori RAM komputer  secara signifikan dan mempercepat proses komputasi saat model Machine Learning dilatih nanti.

Hasil Akhir:
Sekarang  memiliki dua variabel baru, yaitu X_train_icsdf dan X_test_icsdf, yang merupakan matriks fitur berbobot gabungan.

In [84]:
X_train_icsdf = X_train_tfidf.multiply(
    icsdf
)

X_test_icsdf = X_test_tfidf.multiply(
    icsdf
)

In [85]:
X_train_icsdf = csr_matrix(
    X_train_icsdf
)

X_test_icsdf = csr_matrix(
    X_test_icsdf
)

## 18. Menentukan kata paling penting

kode tersebut berfungsi untuk melakukan seleksi fitur (feature selection) dengan cara mencari 100 kata paling penting berdasarkan rata-rata nilai bobot gabungan (TF-IDF × ICSDF) terbesar pada data latih, lalu menampilkan 30 kata teratas dalam bentuk tabel.
Berikut adalah penjelasan detail dari setiap baris kodenya:
## 1. Menghitung Rata-rata Skor Fitur (skor_fitur = ...)

* X_train_icsdf.mean(axis=0): Menghitung nilai rata-rata (mean) dari bobot gabungan untuk setiap kata (kolom) di seluruh dokumen (baris). axis=0 berarti perhitungan dilakukan secara vertikal per kolom.
* np.asarray(...).ravel(): Mengubah hasil rata-rata yang awalnya berbentuk matriks 2D menjadi array 1 dimensi biasa agar nilainya mudah diolah.

## 2. Mengambil 100 Kata Teratas (TOP_K = 100 & top_index = ...)

* np.argsort(skor_fitur): Mengurutkan indeks kata berdasarkan skornya dari yang terkecil ke terbesar.
* [::-1]: Membalikkan urutan hasil argsort agar indeks kata bergeser dari yang terbesar ke terkecil (skor tertinggi berada di paling depan).
* [:TOP_K]: Memotong (slicing) daftar indeks tersebut untuk mengambil 100 indeks kata terbaik saja.
* kata_penting = features[top_index]: Mengambil nama kata asli dari variabel features (kosakata asli) berdasarkan 100 indeks terbaik yang sudah disaring.

## 3. Menampilkan Sampel Kata Penting

* kata_penting[:30]: Menampilkan 30 nama kata teratas dalam bentuk array teks biasa.

## 4. Membuat dan Menampilkan DataFrame (df_kata_penting = ...)

* pd.DataFrame(...): Menyusun 100 kata penting beserta nilai skor rata-ratanya (skor_fitur[top_index]) ke dalam sebuah tabel DataFrame Pandas agar rapi dan mudah dibaca.
* df_kata_penting.head(30): Menampilkan 30 baris pertama dari tabel kata penting tersebut sebagai sampel di layar




In [86]:
skor_fitur = np.asarray(
    X_train_icsdf.mean(
        axis=0
    )
).ravel()

In [87]:
TOP_K = 100

In [88]:
top_index = np.argsort(
    skor_fitur
)[::-1][:TOP_K]

In [89]:
kata_penting = features[
    top_index
]

kata_penting[:30]

array(['marquez', 'false', 'saham', 'balapan', 'purbaya', 'motogp',
       'bandara', 'emas', 'olahraga', 'marc', 'bezzecchi', 'set', 'pupuk',
       'bilal', 'reza', 'aset', 'harga', 'ufc', 'rp', 'sabar', 'aragon',
       'artikel', 'ethan', 'saya', 'rekening', 'bagnaia', 'padel',
       'asian', 'veda', 'kapal'], dtype=object)

In [90]:
df_kata_penting = pd.DataFrame({

    "kata": kata_penting,

    "skor": skor_fitur[
        top_index
    ]
})

df_kata_penting.head(30)

,kata,skor
0,marquez,0.055571
1,false,0.054775
2,saham,0.052213
3,balapan,0.049462
4,purbaya,0.048462
5,motogp,0.045814
6,bandara,0.043981
7,emas,0.042182
8,olahraga,0.041307
9,marc,0.041243


## 19. Reduksi menjadi 100 fitur ICSDF

kode tersebut berfungsi untuk menerapkan seleksi fitur (feature selection) secara nyata pada matriks data. Kode ini memotong matriks fitur (X_train dan X_test) sehingga hanya menyimpan kolom-kolom dari 100 kata paling penting (TOP_K = 100) yang sudah di cari pada tahap sebelumnya, lalu mencetak perbandingan ukuran dimensinya.
Berikut adalah penjelasan detail dari alur kodenya:
## 1. Memotong Kolom Matriks Fitur (_selected = ...)

* X_train_icsdf[:, top_index]: Melakukan pemotongan (slicing) pada matriks data latih. Tanda titik dua (:) di awal berarti kita mengambil semua baris dokumen (berita) tanpa dikurangi, sedangkan top_index berarti kita hanya memilih kolom kata yang masuk ke dalam daftar 100 kata terbaik berbobot ICSDF tinggi.
* X_test_icsdf[:, top_index]: Proses pemotongan yang sama persis diterapkan pada matriks data uji menggunakan indeks yang sama dari data latih (untuk menjaga konsistensi fitur).

## 2. Menampilkan Perbandingan Dimensi (print(...))

* X_train_tfidf.shape: Menampilkan dimensi matriks sebelum diseleksi. Formatnya adalah (jumlah_dokumen, jumlah_seluruh_kata).
* X_train_selected.shape: Menampilkan dimensi matriks setelah diseleksi. Formatnya akan menjadi (jumlah_dokumen, 100).

-

In [91]:
X_train_selected = X_train_icsdf[
    :,
    top_index
]

X_test_selected = X_test_icsdf[
    :,
    top_index
]

In [92]:
print(
    "Sebelum seleksi:",
    X_train_tfidf.shape
)

print(
    "Sesudah ICSDF:",
    X_train_selected.shape
)

Sebelum seleksi: (160, 3069)
Sesudah ICSDF: (160, 100)


## 20. Tabel hasil ICSDF

kode tersebut berfungsi untuk mengubah matriks hasil seleksi fitur (Top 100 kata penting) menjadi sebuah DataFrame Pandas yang rapi, memasukkan label kategori beritanya, lalu menampilkan 5 sampel data pertama.

In [93]:
icsdf_train_df = pd.DataFrame(

    X_train_selected.toarray(),

    columns=kata_penting
)

icsdf_train_df["label"] = (

    y_train.reset_index(
        drop=True
    )
)

icsdf_train_df.head()

,marquez,false,saham,balapan,purbaya,motogp,bandara,emas,olahraga,marc,...,kendaraan,masing,bpa,perjalanan,bantuan,kan,pertashop,puan,pemain,label
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,...,0.0,0.000000,0.0,0.128611,0.0,0.0,0.0,0.0,0.0,0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.23744,0.0,...,0.0,0.180899,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,1
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,...,0.0,0.000000,0.0,0.917834,0.0,0.0,0.0,0.0,0.0,0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,...,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,1
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,...,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,1


In [94]:
icsdf_train_df.to_excel(
    "data_icsdf_training.xlsx",
    index=False
)

## 21. PCA

kode tersebut berfungsi untuk melakukan reduksi dimensi tingkat lanjut menggunakan metode PCA (Principal Component Analysis). Tujuannya adalah memeras informasi dari 100 fitur kata penting sebelumnya menjadi hanya 20 komponen utama (principal components).
Berikut adalah penjelasan detail dari setiap tahapan kodenya:
## 1. Inisialisasi PCA (pca = PCA(...))

* n_components=20:  menentukan secara spesifik bahwa dimensi data baru akan diperkecil menjadi 20 kolom/fitur saja. PCA akan mengekstrak kombinasi linear dari fitur-fitur lama yang paling banyak mempertahankan variansi (informasi) data.
* random_state=42: Menjamin agar proses kalkulasi matematis berbasis matriks di dalam PCA selalu menghasilkan komponen yang sama persis setiap kali kode dijalankan ulang (reproducible).

## 2. Konversi ke Matriks Padat (_dense = ...)

* X_train_selected.toarray(): Mengubah matriks renggang (sparse matrix) menjadi array numerik padat (dense array). Langkah ini wajib dilakukan karena pustaka PCA di scikit-learn secara umum memerlukan input berupa matriks padat (dense).

## 3. Ekstraksi Komponen Utama (fit_transform & transform)

* pca.fit_transform(X_train_selected_dense): PCA akan mempelajari pola struktur data latih, menghitung arah variansi terbesar (vektor eigen), lalu mentransformasikan 100 kolom kata menjadi 20 kolom komponen baru. Hasilnya disimpan di X_train_pca.
* pca.transform(X_test_selected_dense): Mengubah data uji menggunakan parameter yang sudah dipelajari dari data latih. Ingat, jangan gunakan fit pada data uji untuk menghindari kebocoran data (data leakage).

## 4. Menampilkan Dimensi Baru (print(...))

* X_train_pca.shape: Akan menghasilkan output (jumlah_dokumen_latih, 20).
* X_test_pca.shape: Akan menghasilkan output (jumlah_dokumen_uji, 20).

------------------------------
## Perbedaan Penting Seleksi Fitur vs PCA:

* Pada tahap sebelumnya (Top 100), melakukan Seleksi Fitur (memilih 100 kata asli dan membuang sisanya). Kolomnya masih berupa kata nyata (seperti "hoaks", "vaksin").
* Pada tahap ini, melakukan Ekstraksi Fitur melalui PCA. Komponen baru (PC1, PC2, dst.) sudah bukan berupa kata lagi, melainkan kombinasi angka matematis yang merepresentasikan ringkasan informasi dari 100 kata tersebut.






In [95]:
pca = PCA(

    n_components=20,

    random_state=42
)

In [96]:
X_train_selected_dense = (
    X_train_selected.toarray()
)

X_test_selected_dense = (
    X_test_selected.toarray()
)

In [97]:
X_train_pca = pca.fit_transform(
    X_train_selected_dense
)

X_test_pca = pca.transform(
    X_test_selected_dense
)

In [98]:
print(
    X_train_pca.shape
)

print(
    X_test_pca.shape
)

(160, 20)
(40, 20)


## 22. Membuat tabel data reduksi training

 kode tersebut berfungsi untuk menyusun hasil reduksi dimensi PCA ke dalam DataFrame Pandas yang terstruktur, memberikan nama kolom yang representatif untuk setiap komponen utama, memasukkan label target, lalu menampilkan sampel datanya.

In [99]:
nama_pc = [
    f"PC{i}"
    for i in range(
        1,
        21
    )
]

In [100]:
df_train_reduksi = pd.DataFrame(

    X_train_pca,

    columns=nama_pc
)

df_train_reduksi["label"] = (

    y_train.reset_index(
        drop=True
    )
)

df_train_reduksi.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,label
0,-0.037879,0.020732,-0.075568,0.057915,-0.035389,-0.008722,-0.031845,-0.033064,-0.005557,0.032311,...,-0.020244,0.022448,-0.002366,-0.043881,-0.029974,-0.026482,0.005551,-0.006335,0.023289,0
1,-0.039132,-0.018399,-0.058630,-0.007808,-0.069948,0.016156,-0.075881,-0.041786,-0.056430,0.102206,...,-0.061329,0.065043,-0.000269,-0.133691,-0.089575,-0.095962,0.007548,-0.028212,0.054110,1
2,-0.036714,-0.015158,-0.055574,0.005273,-0.054974,0.014632,-0.066286,0.003891,-0.041890,0.039326,...,-0.053736,0.056716,-0.010340,-0.104318,-0.068913,-0.100407,0.013257,0.071497,-0.012938,0
3,-0.053259,-0.019859,-0.124056,-0.074777,-0.085109,0.012057,-0.131203,-0.183253,-0.741277,-0.541490,...,0.196056,-0.058088,-0.156856,0.348167,-0.081540,0.068128,-0.009295,-0.028324,0.004711,1
4,-0.038427,-0.024738,-0.076596,-0.026420,-0.097193,0.317350,0.172879,0.016220,0.003298,0.006180,...,-0.021865,0.022536,-0.006739,-0.034129,-0.020453,-0.013665,0.002255,-0.023234,0.022042,1


## 23. Data testing

In [101]:
df_test_reduksi = pd.DataFrame(

    X_test_pca,

    columns=nama_pc
)

df_test_reduksi["label"] = (

    y_test.reset_index(
        drop=True
    )
)

df_test_reduksi.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,label
0,-0.050450,-0.023579,-0.129063,-0.003784,-0.164909,-0.000443,-0.273660,0.513483,0.042071,0.012307,...,-0.041993,0.067907,-0.048563,-0.171754,-0.116831,-0.350094,0.027037,0.512247,-0.255672,0
1,-0.057598,-0.043956,-0.164615,-0.114014,-0.053901,0.020612,-0.174186,-0.218758,-0.902092,-0.659224,...,0.210326,0.065742,-0.441488,0.505872,-0.110165,0.053360,-0.005368,0.016431,-0.030886,1
2,-0.091238,-0.563481,1.007166,0.158704,0.372247,-0.076150,0.176333,0.063060,0.102566,-0.034249,...,0.084905,-0.086995,0.017680,0.134932,0.065794,0.070648,-0.008268,0.081929,-0.180263,1
3,-0.037750,-0.023789,-0.073339,-0.025009,-0.092281,0.295030,0.159605,0.016014,0.002389,0.004895,...,-0.027416,0.023598,-0.005902,-0.037275,-0.021555,-0.013835,0.002190,-0.022350,0.024320,1
4,-0.039992,0.017435,-0.063858,0.017269,-0.057008,0.003475,-0.073026,-0.002314,-0.031900,0.065919,...,-0.253114,-0.108382,-0.000434,0.085542,0.046686,0.012270,-0.000870,0.009127,-0.006571,0


## 24. Simpan data reduksi

In [102]:
df_train_reduksi.to_excel(
    "data_reduksi_training.xlsx",
    index=False
)

df_test_reduksi.to_excel(
    "data_reduksi_testing.xlsx",
    index=False
)

## 25. Cek total training dan testing

In [103]:
print(
    "Training :",
    df_train_reduksi.shape
)

print(
    "Testing :",
    df_test_reduksi.shape
)

Training : (160, 21)
Testing : (40, 21)


## 26. Kolom terakhir harus label

In [104]:
df_train_reduksi.columns

Index(['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10',
       'PC11', 'PC12', 'PC13', 'PC14', 'PC15', 'PC16', 'PC17', 'PC18', 'PC19',
       'PC20', 'label'],
      dtype='object')